# 🚀 Live Data Explorer

**Lumexa Python Foundations · Course 3: Python Tools & Data Foundations · Lesson 7**

| | |
|---|---|
| **Project** | Live Data Explorer (portfolio project 8 of 9) |
| **You will practise** | `requests`, HTTP status codes, JSON, error handling, lists of dictionaries, Pandas DataFrames, API safety |
| **Time** | about 60 minutes |

Ask a real API, "**Where is the International Space Station right now?**", and turn the answer into a neat report. You will make requests, check status codes, read JSON, handle errors, collect several readings in a table and save them to a file.

## 🧭 How to use this notebook

1. **Keep your own copy.** Choose **File → Save a copy in Drive** first, so your work is saved in *your* Google Drive.
2. **Run the cells in order**, from the top, with **Shift + Enter** (or **Runtime → Run all**). Read the text above each cell before you run it.
3. Cells marked **✏️ Your turn** are for you to change or complete. They are safe to run before you finish them.
4. If something goes wrong, use **Runtime → Restart session and run all**. A notebook you can run from the top is a notebook you can trust.

> 🌐 This notebook talks to a live service on the internet. If the service cannot be reached, the notebook uses a **saved sample** (and says so), so it never breaks. No account or API key is needed.

## ⚙️ Setup

`requests` is the library that sends HTTP requests. (It is already installed in Google Colab. On your own computer, add it with `python -m pip install requests`.)

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

ISS_URL = "https://api.wheretheiss.at/v1/satellites/25544"
print("requests version:", requests.__version__)

## Part 1 · Meet the API

**What:** "Where the ISS at?" is a free public API that tells you the position of the International Space Station.
**Address (URL):** `https://api.wheretheiss.at/v1/satellites/25544` (25544 is the satellite number of the ISS).
**Key needed?** No.
**Rate limit:** the service asks for roughly **one request per second at most**, so we will pause between requests. Be a polite visitor!

> 🔐 **Safety first.** This API needs no key. If you ever use an API that does, **never paste the key into a notebook you share**. Use Colab's **Secrets** panel (the 🔑 icon on the left) or an environment variable instead, and never send personal information to an API.

### The conversation

1. Your program (the **client**) sends a **GET request** to the URL.
2. The **server** sends back a **response** with a **status code** (200 means *OK*) and the data as **JSON** text.
3. `response.json()` turns the JSON text into Python dictionaries and lists.

## Part 2 · A careful request function

Networks fail, servers have bad days and addresses get mistyped. So we put the request in a function that:

- sets a **timeout**, so the program never waits forever,
- catches **network errors** with `try` and `except`,
- checks the **status code** before it uses the data,
- returns a dictionary, or `None` when something went wrong.

In [ ]:
def fetch_json(url):
    """Ask an API for JSON. Return a dictionary, or None if anything goes wrong."""
    try:
        response = requests.get(url, timeout=10)
    except requests.exceptions.RequestException as error:
        print("❌ Could not reach the API:", type(error).__name__)
        return None
    if response.status_code != 200:
        print(f"❌ The API answered with status code {response.status_code}.")
        return None
    return response.json()

data = fetch_json(ISS_URL)
print("Got data!" if data else "No data this time (we will use a saved sample).")

## Part 3 · Read the response

Print the **whole response** first and read it carefully. It is a dictionary, so you can see every key. We keep a **saved sample** of a real response so the rest of the notebook works even without the internet.

In [ ]:
SAMPLE_RESPONSE = {
    "name": "iss", "id": 25544, "latitude": 47.4637, "longitude": -137.5851,
    "altitude": 423.67, "velocity": 27587.33, "visibility": "eclipsed",
    "footprint": 4526.08, "timestamp": 1794558703, "units": "kilometers",
}

def get_reading(step=0):
    """Return (data, source). The source is 'live', or 'sample' if the API could not be reached."""
    data = fetch_json(ISS_URL)
    if data is not None:
        return data, "live"
    fake = dict(SAMPLE_RESPONSE)                              # a copy of the saved sample
    fake["latitude"] = round(fake["latitude"] + step * 0.9, 4)     # pretend the station moved
    fake["longitude"] = round(fake["longitude"] + step * 1.3, 4)
    fake["timestamp"] = fake["timestamp"] + step * 2
    return fake, "sample"

data, source = get_reading()
print("Source of this reading:", source)
print(json.dumps(data, indent=2))

✏️ **Your turn:** Look at the response above and answer in a Markdown cell: *How many keys does it have? Which keys hold numbers, and which hold text? What do you think `footprint` means?*

## Part 4 · Pick out the facts and show a report

Now we extract the fields we want and format them with **f-strings**. Two small helpers make the output friendlier: one turns a latitude and longitude into words, and one turns the Unix `timestamp` (seconds since 1970) into a readable date and time.

In [ ]:
def describe_position(latitude, longitude):
    north_south = "north" if latitude >= 0 else "south"
    east_west = "east" if longitude >= 0 else "west"
    return f"{abs(latitude):.1f}° {north_south}, {abs(longitude):.1f}° {east_west}"

def readable_time(timestamp):
    moment = datetime.fromtimestamp(timestamp, tz=timezone.utc)
    return moment.strftime("%Y-%m-%d %H:%M:%S UTC")

def show_report(data, source="live"):
    print("🛰️  INTERNATIONAL SPACE STATION")
    print(f"   Time:       {readable_time(data['timestamp'])}")
    print(f"   Position:   {describe_position(data['latitude'], data['longitude'])}")
    print(f"   Altitude:   {data['altitude']:.1f} {data['units']}")
    print(f"   Speed:      {data['velocity']:,.0f} km/h")
    print(f"   Visibility: {data['visibility']}")
    map_link = f"https://www.google.com/maps?q={data['latitude']},{data['longitude']}"
    print(f"   Map:        {map_link}")
    print(f"   (data source: {source})")

show_report(data, source)

## Part 5 · Three readings, a few seconds apart

The station moves fast, so let's **measure** it. We ask for three readings, two seconds apart (`time.sleep(2)` pauses the program), store each one as a **dictionary** in a **list**, and turn the list into a Pandas **DataFrame**.

In [ ]:
readings = []
for step in range(3):
    data, source = get_reading(step)
    readings.append({
        "time_utc": readable_time(data["timestamp"]),
        "timestamp": data["timestamp"],
        "latitude": data["latitude"],
        "longitude": data["longitude"],
        "altitude_km": round(data["altitude"], 1),
        "speed_kmh": round(data["velocity"]),
        "visibility": data["visibility"],
        "source": source,
    })
    print(f"Reading {step + 1} of 3 collected ({source})")
    if step < 2:
        time.sleep(2)          # be polite: do not send requests too fast

table = pd.DataFrame(readings)
table[["time_utc", "latitude", "longitude", "altitude_km", "speed_kmh", "visibility", "source"]]

In [ ]:
# How far did the station move between the first and the last reading?
seconds = table["timestamp"].iloc[-1] - table["timestamp"].iloc[0]
km_travelled = table["speed_kmh"].mean() * seconds / 3600
print(f"Time between the first and last reading: {seconds} seconds")
print(f"Latitude changed by {table['latitude'].iloc[-1] - table['latitude'].iloc[0]:+.2f}°")
print(f"Longitude changed by {table['longitude'].iloc[-1] - table['longitude'].iloc[0]:+.2f}°")
print(f"At about {table['speed_kmh'].mean():,.0f} km/h, that is roughly {km_travelled:.0f} km of travel.")

## Part 6 · When things go wrong

A good program **expects errors**. Here we ask for two things that cannot work, and watch `fetch_json()` handle both politely instead of crashing:

- an address on the right server that does not exist (status code **404**),
- a server that does not exist at all (a **connection error**).

In [ ]:
print("1) A satellite that does not exist:")
result = fetch_json("https://api.wheretheiss.at/v1/satellites/0")
print("   returned:", result)

print("\n2) A server that does not exist:")
result = fetch_json("https://this-server-does-not-exist.invalid/")
print("   returned:", result)

print("\nThe program is still running! 🎉")

### Status codes you will meet

| Code | Meaning | What to do |
|---|---|---|
| **200** | OK | Use the data |
| **400** | Bad request | Check the URL and parameters |
| **401 / 403** | Not allowed | Check your API key |
| **404** | Not found | Check the address |
| **429** | Too many requests | Wait and slow down |
| **500+** | Server problem | Try again later |

## Part 7 · Saving the readings

We save the readings to a **JSON file** with `json.dump()` and read them back with `json.load()`, so the data survives after the notebook closes.

In [ ]:
folder = Path("live_data")
folder.mkdir(exist_ok=True)
file_path = folder / "iss_readings.json"

with open(file_path, "w") as file:
    json.dump(readings, file, indent=2)
print("Saved", len(readings), "readings to", file_path)

with open(file_path) as file:
    loaded = json.load(file)
print("Loaded", len(loaded), "readings back. First one was taken at", loaded[0]["time_utc"])

## ✏️ Your turn · Add a feature

Complete `speed_in_km_per_second()`. It receives a speed in **km/h** and returns the speed in **km per second** (there are 3,600 seconds in an hour). Then use it in a sentence about the ISS.

In [ ]:
def speed_in_km_per_second(speed_kmh):
    return None   # <- write your formula here

print("✅ It works!" if speed_in_km_per_second(3600) == 1 else "⬜ speed_in_km_per_second is not finished yet")

In [ ]:
# ✏️ Your turn: use your function to print a sentence such as
# "The ISS travels about 7.7 kilometres every second!"

## 🏆 Stretch challenges

1. Take **ten** readings, one every two seconds, and find the highest and lowest `altitude_km`.
2. Choose a **second free API with no key** (ask your teacher to approve it). Write `fetch_json()` for it, print the whole response, and extract six fields.
3. Use `params={"units": "miles"}` with `requests.get(ISS_URL, params=..., timeout=10)` and compare the two answers. (The ISS API accepts `units=miles`.)
4. Draw a chart of `latitude` against `longitude` for your readings with `table.plot(x="longitude", y="latitude", kind="scatter")`.

## 🪞 Reflection

- Why is it important to check the status code before using the data?
- What could go wrong if 300 students all ran a loop of requests with no pause?
- What would you do if you saw a classmate's API key in a notebook they were sharing?

*Your answers:*

## 📤 How to submit

Make sure the notebook runs from the top with **Runtime → Restart session and run all**. Then choose **File → Save a copy in Drive**, **Share → Anyone with the link can view**, and send the link to your teacher.